# DATA PRE-PROCESSING

In [1]:
import pandas as pd
df = pd.read_csv("AMJATH.txt",header=None,on_bad_lines='skip',encoding='utf8')
df

,0,1,2,3
0,4/5/22,6:31 PM - Messages and calls are end-to-end e...,listen to,or share them. Learn more.
1,4/5/22,6:31 PM - Amjath Kncet changed their phone nu...,NaN,NaN
2,4/5/22,8:09 PM - AMJATH: Hi da I'm amjath,NaN,NaN
3,4/5/22,8:09 PM - AMJATH: My new number,NaN,NaN
4,4/6/22,7:01 AM - priyan: 👍,NaN,NaN
...,...,...,...,...
3883,11/1/25,1:45 PM - priyan: https://www.1tamilmv.farm/i...,NaN,NaN
3884,11/1/25,1:45 PM - priyan: https://www.1tamilmv.farm/i...,NaN,NaN
3885,11/1/25,1:45 PM - priyan: https://www.1tamilmv.farm/i...,NaN,NaN
3886,11/1/25,1:45 PM - priyan: https://www.1tamilmv.farm/i...,NaN,NaN


In [2]:
df=df.drop(0)
df=df.drop(1)
df.drop([2, 3], axis=1, inplace=True)
df

,0,1
2,4/5/22,8:09 PM - AMJATH: Hi da I'm amjath
3,4/5/22,8:09 PM - AMJATH: My new number
4,4/6/22,7:01 AM - priyan: 👍
5,5/24/22,10:56 AM - AMJATH: <Media omitted>
6,8/12/22,4:47 PM - AMJATH: http://tpcg.io/_YE2YX0
...,...,...
3883,11/1/25,1:45 PM - priyan: https://www.1tamilmv.farm/i...
3884,11/1/25,1:45 PM - priyan: https://www.1tamilmv.farm/i...
3885,11/1/25,1:45 PM - priyan: https://www.1tamilmv.farm/i...
3886,11/1/25,1:45 PM - priyan: https://www.1tamilmv.farm/i...


In [3]:
df.columns=['Date','Chat']
Message=df['Chat'].str.split('-',n=1,expand=True)
df['Time']=Message[0]
Message1=Message[1].str.split(':',n=1,expand=True)
df['Name']=Message1[0]
df['Chat']=Message1[1]
df=df[['Date','Time','Name','Chat']]
df

,Date,Time,Name,Chat
2,4/5/22,8:09 PM,AMJATH,Hi da I'm amjath
3,4/5/22,8:09 PM,AMJATH,My new number
4,4/6/22,7:01 AM,priyan,👍
5,5/24/22,10:56 AM,AMJATH,<Media omitted>
6,8/12/22,4:47 PM,AMJATH,http://tpcg.io/_YE2YX0
...,...,...,...,...
3883,11/1/25,1:45 PM,priyan,https://www.1tamilmv.farm/index.php?/forums/t...
3884,11/1/25,1:45 PM,priyan,https://www.1tamilmv.farm/index.php?/forums/t...
3885,11/1/25,1:45 PM,priyan,https://www.1tamilmv.farm/index.php?/forums/t...
3886,11/1/25,1:45 PM,priyan,https://www.1tamilmv.farm/index.php?/forums/t...


# SENTIMENTAL ANALYSIS

In [4]:
data=df
data

,Date,Time,Name,Chat
2,4/5/22,8:09 PM,AMJATH,Hi da I'm amjath
3,4/5/22,8:09 PM,AMJATH,My new number
4,4/6/22,7:01 AM,priyan,👍
5,5/24/22,10:56 AM,AMJATH,<Media omitted>
6,8/12/22,4:47 PM,AMJATH,http://tpcg.io/_YE2YX0
...,...,...,...,...
3883,11/1/25,1:45 PM,priyan,https://www.1tamilmv.farm/index.php?/forums/t...
3884,11/1/25,1:45 PM,priyan,https://www.1tamilmv.farm/index.php?/forums/t...
3885,11/1/25,1:45 PM,priyan,https://www.1tamilmv.farm/index.php?/forums/t...
3886,11/1/25,1:45 PM,priyan,https://www.1tamilmv.farm/index.php?/forums/t...


In [5]:
 def sentimentalAnalysis(data,columnname):
        from nltk.sentiment.vader import SentimentIntensityAnalyzer
        data.dropna(inplace=True)
        sid = SentimentIntensityAnalyzer()
        data = data.dropna(subset=[columnname])
        data['scores'] = data[columnname].apply(lambda commentText: sid.polarity_scores(commentText))
        data['compound']  = data['scores'].apply(lambda score_dict: score_dict['compound'])
        data['Negtive']  = data['scores'].apply(lambda score_dict: score_dict['neg'])
        data['Postive']  = data['scores'].apply(lambda score_dict: score_dict['pos'])
        data['Neutral']  = data['scores'].apply(lambda score_dict: score_dict['neu'])
        data['comp_score'] = data['compound'].apply(lambda c: 'pos' if c >=0 else 'neg')
        posneg=pd.DataFrame(data['comp_score'].value_counts())
        return posneg,data

In [6]:
pos,data_Senti=sentimentalAnalysis(data,columnname='Chat')

/var/folders/qc/j1zhz5wd799gnfl61hmhl4kw0000gn/T/ipykernel_2002/2738220752.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data.dropna(inplace=True)


In [7]:
pos

,count
comp_score,
pos,3703
neg,69


# TOPIC MODELLING

In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [9]:
tfidf=TfidfVectorizer(max_df=0.95,min_df=2,stop_words='english')
dtm=tfidf.fit_transform(df["Chat"])

In [10]:
from sklearn.decomposition import NMF
nmf_model=NMF(n_components=5,random_state=42)
nmf_model.fit(dtm)

/opt/anaconda3/lib/python3.13/site-packages/sklearn/decomposition/_nmf.py:1742: ConvergenceWarning: Maximum number of iterations 200 reached. Increase it to improve convergence.
  warnings.warn(


NMF(n_components=5, random_state=42)

In [11]:
for index, topic in enumerate(nmf_model.components_):
    feature_names = tfidf.get_feature_names_out()
    results = [feature_names[i] for i in topic.argsort()[-10:]]
    print(results)

['athan', 'crs', 'potu', 'telugu', 'ivunga', 'ww', 'yae', 'podraga', 'media', 'omitted']
['sunnewstamil', 'iammoviebuff007', 'cinemawithab', 'karthikravivarm', 'mythriofficial', 'letscinema', 'https', 'com', '08', 'status']
['youtu', 'mtc4mmm1ymi2ng', 'mdjmnzvkmjy', 'igshid', 'com', 'https', 'reel', 'igsh', 'www', 'instagram']
['la', 'anuppu', 'illa', 'dei', 'enna', 'ok', 'ama', 'seri', 'hm', 'da']
['itsdilli0700', 'trollywoodx', 'vcdtweets', 'saloon_kada', 'offl_rao', 'https', 'com', 'status', '08', 'twitter']


In [12]:
topic_results=nmf_model.transform(dtm)
df["Topic"]=topic_results.argmax(axis=1)
df

/var/folders/qc/j1zhz5wd799gnfl61hmhl4kw0000gn/T/ipykernel_2002/568022287.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["Topic"]=topic_results.argmax(axis=1)


,Date,Time,Name,Chat,Topic
2,4/5/22,8:09 PM,AMJATH,Hi da I'm amjath,3
3,4/5/22,8:09 PM,AMJATH,My new number,3
4,4/6/22,7:01 AM,priyan,👍,0
5,5/24/22,10:56 AM,AMJATH,<Media omitted>,0
6,8/12/22,4:47 PM,AMJATH,http://tpcg.io/_YE2YX0,2
...,...,...,...,...,...
3883,11/1/25,1:45 PM,priyan,https://www.1tamilmv.farm/index.php?/forums/t...,2
3884,11/1/25,1:45 PM,priyan,https://www.1tamilmv.farm/index.php?/forums/t...,2
3885,11/1/25,1:45 PM,priyan,https://www.1tamilmv.farm/index.php?/forums/t...,2
3886,11/1/25,1:45 PM,priyan,https://www.1tamilmv.farm/index.php?/forums/t...,2


In [13]:
df["Topic"].value_counts()

Topic
3    1486
0    1092
1     695
4     324
2     175
Name: count, dtype: int64

# WORD CLOUD

In [14]:
dataset=df
from wordcloud import WordCloud
from nltk.corpus import stopwords
import nltk
import matplotlib.pyplot as plt
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/srimohanapriyan/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [ ]:
comment_words = []
stoplist = stopwords.words('english')
stoplist.extend(['omitted', 'voice','missed','call','video','deleted','media','message'])
wordcloudss="This function saves image"
dataset.index=range(dataset.shape[0])
for i in range(1,len(dataset)): 
    comment_words.append(dataset['Chat'][i])
    vv=" ".join(comment_words)          
    wordcloud = WordCloud(width = 800, height = 800, 
                                background_color ='white', 
                                      stopwords = stoplist, 
                                      min_font_size = 10).generate(vv)         
plt.figure(figsize = (9, 7), facecolor = None)
plt.imshow(wordcloud)
plt.axis("off") 
plt.tight_layout(pad = 0) 
plt.savefig('wordcloud.PNG')
plt.show() 
print("Successfully created")
wordcloudss="This function saves image"